# **Phase 2 — Clean & Prepare (Steps 9–17)**
Organize columns, clean values, and export a final clean dataset.

In [3]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

%matplotlib inline
sns.set_style("whitegrid")
plt.rcParams["figure.figsize"] = (9, 5)
pd.set_option("display.max_columns", 50)

RAW_PATH = "/content/cityflo_bus_service_metro_cities.csv"
df = pd.read_csv(RAW_PATH)
df.head()
df.shape

(3258, 36)

In [4]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [5]:
df=pd.read_csv("/content/cityflo_bus_service_metro_cities.csv")

# **Step 9 — Split Columns: Numerical vs Categorical**
Separate columns into numerical and categorical groups — they need different analysis and cleaning approaches. (Split is based on intended type, since several numeric columns are still stored as text at this point.)

In [6]:
numerical_cols = ["age", "distance_km", "fare_inr", "discount_inr", "rating", "occupancy_pct"]
categorical_cols = ["gender", "city", "bus_type", "payment_mode", "booking_channel",
                     "trip_status", "cancellation_reason", "weather", "subscription_type",
                     "device_type", "is_peak_hour", "gps_enabled", "complaint_raised"]
datetime_cols = ["trip_date", "scheduled_departure", "actual_departure",
                  "scheduled_arrival", "actual_arrival"]
id_cols = ["trip_id", "booking_id", "customer_id", "route_id", "driver_id", "bus_number"]

print("Numerical:", numerical_cols)
print("\nCategorical:", categorical_cols)
print("\nDate/Time:", datetime_cols)
print("\nID columns:", id_cols)

Numerical: ['age', 'distance_km', 'fare_inr', 'discount_inr', 'rating', 'occupancy_pct']

Categorical: ['gender', 'city', 'bus_type', 'payment_mode', 'booking_channel', 'trip_status', 'cancellation_reason', 'weather', 'subscription_type', 'device_type', 'is_peak_hour', 'gps_enabled', 'complaint_raised']

Date/Time: ['trip_date', 'scheduled_departure', 'actual_departure', 'scheduled_arrival', 'actual_arrival']

ID columns: ['trip_id', 'booking_id', 'customer_id', 'route_id', 'driver_id', 'bus_number']


# **Step 10 — describe() of Numerical Columns**
Run df.describe() to see count, mean, std, min, quartiles, and max. Note: fare_inr still has mixed text formatting at this stage, so we coerce it to numeric first just for this preview (the permanent fix happens in Step 16).

In [7]:
df.describe()

,age,distance_km,discount_inr,rating,occupancy_pct
count,3194.000000,3258.000000,3074.000000,2183.000000,2878.000000
mean,29.477145,24.696071,14.365973,3.798443,58.313343
std,8.251467,10.311346,31.070065,1.163396,25.073318
min,-5.000000,8.300000,0.000000,1.000000,15.100000
25%,23.000000,16.000000,0.000000,3.000000,36.700000
50%,29.000000,25.700000,0.000000,4.000000,58.500000
75%,35.000000,33.100000,8.000000,5.000000,79.500000
max,130.000000,41.800000,203.000000,5.000000,138.800000


# **Step 11 — Write a Summary of the Dataset**

***Summary***: This is a synthetic, trip-level dataset simulating a premium AC bus service (CityFlo-style) operating across six Indian metro cities — Mumbai, Pune, Bangalore, Hyderabad, Chennai, and Delhi NCR. Each of the 3,258 rows represents one passenger trip booked across 2024, capturing customer demographics, route/bus details, scheduled vs. actual timings, fare and payment information, occupancy, ratings, and trip outcomes (completed, delayed, cancelled, or no-show).

***Source***: Synthetically generated for EDA-workflow practice (not real operator data).

***Size:*** 3,258 rows × 36 columns (~882 KB), including 58 intentional exact-duplicate rows.

***General quality:*** Deliberately "raw" — it contains inconsistent category labels (e.g. Male/M/male), mixed boolean representations (Yes/No/1/0/True/False), mixed date formats, currency-symbol/comma inconsistencies in fare_inr, stray whitespace, missing values in several columns (mostly context-dependent, e.g. no rating for cancelled trips), and a handful of outliers / data-entry errors (negative ages, >100% occupancy, extreme fares).

In [8]:
print(f"Rows: {df.shape[0]:}  |  Columns: {df.shape[1]}")
print(f"Cities covered: {sorted(df['city'].str.strip().unique())}")
print(f"Date range (raw, unparsed sample): {df['trip_date'].min()} .. {df['trip_date'].max()}")
print(f"Trip status breakdown:\n{df['trip_status'].value_counts()}")

Rows: 3258  |  Columns: 36
Cities covered: ['Bangalore', 'Chennai', 'Delhi NCR', 'Hyderabad', 'Mumbai', 'Pune']
Date range (raw, unparsed sample): 01-Feb-2024 .. September 27, 2024
Trip status breakdown:
trip_status
Completed            2148
Delayed-Completed     390
Cancelled             380
No-show               340
Name: count, dtype: int64


# **Step 12 — Write the Problem Statement**
Problem Statement: What factors drive trip delays, cancellations, and low customer satisfaction in CityFlo's metro-city bus operations, and how do demand, revenue, and occupancy vary across cities, routes, bus types, and time (peak vs. non-peak, day of week, month)?

This drives every scoping decision from here on — which columns are required, which derived metrics matter, and which relationships we test statistically.

# **Step 13 — Mention the Columns That Are Required**
Columns needed to answer the problem statement above:

Identifiers (for joins/grouping): trip_id, customer_id, route_id

Where: city, route_name, origin_stop, destination_stop, bus_type

When: trip_date, scheduled_departure, actual_departure, scheduled_arrival, actual_arrival, is_peak_hour

Trip outcome: trip_status, cancellation_reason, rating, occupancy_pct, complaint_raised

Money: distance_km, fare_inr, discount_inr, payment_mode

Context: weather, subscription_type, booking_channel, gps_enabled

Demographics: gender, age

# **Step 14 — Drop the Columns That Are Not Required**

Not needed to answer the problem statement — either purely transactional IDs with no analytic value, or personally identifiable information not required once customer_id is retained for grouping:

booking_id — transactional ID, no analytic value once trip_id exists

customer_name — PII, not needed for aggregate analysis (customer_id retained)

driver_id, driver_name — outside the stated problem's scope (route/city/ops level, not driver-level)

bus_number — redundant once bus_type + city are retained

seat_number — no analytic value

device_type — outside the stated problem's scope

In [9]:
cols_to_drop = ["booking_id", "customer_name", "driver_id", "driver_name",
                 "bus_number", "seat_number", "device_type"]
df = df.drop(columns=cols_to_drop)
print(f"Dropped {len(cols_to_drop)} columns. Remaining: {df.shape[1]}")
df.columns.tolist()

Dropped 7 columns. Remaining: 29


['trip_id',
 'customer_id',
 'gender',
 'age',
 'city',
 'route_id',
 'route_name',
 'origin_stop',
 'destination_stop',
 'bus_type',
 'trip_date',
 'scheduled_departure',
 'actual_departure',
 'scheduled_arrival',
 'actual_arrival',
 'distance_km',
 'fare_inr',
 'discount_inr',
 'payment_mode',
 'booking_channel',
 'trip_status',
 'cancellation_reason',
 'rating',
 'occupancy_pct',
 'weather',
 'is_peak_hour',
 'subscription_type',
 'gps_enabled',
 'complaint_raised']

# **Step 15 — Add Derived Columns If Required**

Useful columns not present in the raw data, created from existing ones:

trip_year, trip_month, trip_weekday — extracted from trip_date once parsed

delay_minutes — actual vs. scheduled departure gap

is_delayed — flag when delay_minutes > 5
net_revenue_inr — fare_inr minus discount_inr

age_group — binned age brackets

(Date parsing and numeric coercion happen properly in Step 16 — derived columns below use the cleaned versions produced there, computed together to avoid redundant parsing.)

# **Step 16 — Perform Cleaning Operations on Each Column**

Applying the right cleaning method based on each column's type and role, as outlined in the checklist: text/categorical standardization, numerical imputation & outlier handling, date/time parsing, ID validation, boolean normalization, and cross-column consistency checks.


16a — Text / Categorical columns:

standardize case, strip whitespace, fix inconsistent labels (e.g. gender variants), normalize boolean-style flag columns to a single representation.

In [10]:
# Strip whitespace from all object columns + headers
df.columns = df.columns.str.strip()
str_cols = df.select_dtypes(include="object").columns
for c in str_cols:
    df[c] = df[c].astype(str).str.strip()
    df.loc[df[c].isin(["", "nan", "None"]), c] = np.nan

# Fix inconsistent gender labels -> Male / Female / Other
gender_map = {
    "male": "Male", "m": "Male", "male ": "Male",
    "female": "Female", "f": "Female",
    "other": "Other", "o": "Other",
}
df["gender"] = df["gender"].str.lower().map(gender_map).fillna(df["gender"])
print(df["gender"].value_counts())

gender
Female    1655
Male      1455
Other      148
Name: count, dtype: int64


In [11]:
# Normalize messy boolean flag columns to real booleans
def to_bool(series):
    true_vals = {"yes", "y", "1", "true"}
    false_vals = {"no", "n", "0", "false"}
    s = series.astype(str).str.strip().str.lower()
    return s.map(lambda v: True if v in true_vals else (False if v in false_vals else np.nan))

for c in ["is_peak_hour", "gps_enabled", "complaint_raised"]:
    df[c] = to_bool(df[c])

df[["is_peak_hour", "gps_enabled", "complaint_raised"]].apply(lambda s: s.value_counts(dropna=False))

,is_peak_hour,gps_enabled,complaint_raised
False,1957,318,3047
True,1301,2940,211


16b — Numerical columns:

type-correct fare_inr (strip ₹ / INR / commas), handle missing values, detect & treat outliers (IQR method), check skewness/kurtosis, and fix impossible values (negative ages, >100% occupancy).

In [12]:
# Type correction: fare_inr stored as text with currency symbols/commas
df["fare_inr"] = (
    df["fare_inr"].astype(str)
    .str.replace("₹", "", regex=False)
    .str.replace("INR", "", regex=False)
    .str.replace(",", "", regex=False)
    .str.strip()
)
df["fare_inr"] = pd.to_numeric(df["fare_inr"], errors="coerce")

for c in ["age", "distance_km", "discount_inr", "rating", "occupancy_pct"]:
    df[c] = pd.to_numeric(df[c], errors="coerce")

df[["age", "distance_km", "fare_inr", "discount_inr", "rating", "occupancy_pct"]].dtypes

,0
age,float64
distance_km,float64
fare_inr,int64
discount_inr,float64
rating,float64
occupancy_pct,float64


In [13]:
# Fix impossible values (domain-based thresholds)
df.loc[(df["age"] < 5) | (df["age"] > 100), "age"] = np.nan
df.loc[(df["occupancy_pct"] < 0) | (df["occupancy_pct"] > 100), "occupancy_pct"] = np.nan
df.loc[df["fare_inr"] <= 0, "fare_inr"] = np.nan

# Outlier detection on fare_inr via IQR method
q1, q3 = df["fare_inr"].quantile([0.25, 0.75])
iqr = q3 - q1
lower, upper = q1 - 1.5 * iqr, q3 + 1.5 * iqr
outliers = df[(df["fare_inr"] < lower) | (df["fare_inr"] > upper)]
print(f"fare_inr IQR bounds: [{lower:.0f}, {upper:.0f}]  |  Outliers flagged: {len(outliers)}")

# Cap (winsorize) rather than drop, to preserve trip records
df["fare_inr"] = df["fare_inr"].clip(lower=lower, upper=upper)

fare_inr IQR bounds: [-158, 766]  |  Outliers flagged: 30


In [14]:
# Skewness & Kurtosis check
for c in ["age", "distance_km", "fare_inr", "occupancy_pct"]:
    print(f"{c:15s} skew={df[c].skew():+.2f}   kurtosis={df[c].kurt():+.2f}")

age             skew=+0.36   kurtosis=-0.43
distance_km     skew=-0.13   kurtosis=-1.14
fare_inr        skew=+0.46   kurtosis=-0.22
occupancy_pct   skew=-0.01   kurtosis=-1.22


In [15]:
# Missing-value imputation (median for numeric, context-aware for rating)
df["age"] = df["age"].fillna(df["age"].median())
df["discount_inr"] = df["discount_inr"].fillna(0)
# rating intentionally stays NaN for cancelled/no-show trips (no rating was ever given)
df["occupancy_pct"] = df.groupby("trip_status")["occupancy_pct"].transform(
    lambda s: s.fillna(s.median())
)

16c — Date / Time columns:

standardize trip_date (mixed formats) with pd.to_datetime, then extract derived parts.

In [16]:
df["trip_date"] = pd.to_datetime(df["trip_date"], format="mixed", dayfirst=True, errors="coerce")

df["trip_year"] = df["trip_date"].dt.year
df["trip_month"] = df["trip_date"].dt.month
df["trip_weekday"] = df["trip_date"].dt.day_name()

print(f"Unparseable trip_date rows: {df['trip_date'].isna().sum()}")
df[["trip_date", "trip_year", "trip_month", "trip_weekday"]].head()

Unparseable trip_date rows: 0


,trip_date,trip_year,trip_month,trip_weekday
0,2024-09-21,2024,9,Saturday
1,2024-08-08,2024,8,Thursday
2,2024-03-19,2024,3,Tuesday
3,2024-12-29,2024,12,Sunday
4,2024-05-15,2024,5,Wednesday


In [17]:
 #Delay in minutes: actual vs scheduled departure (HH:MM strings -> minutes since midnight)
def to_minutes(t):
    if pd.isna(t) or t == "" or str(t).lower() == "nan":
        return np.nan
    h, m = str(t).split(":")
    return int(h) * 60 + int(m)

sched_dep_min = df["scheduled_departure"].apply(to_minutes)
actual_dep_min = df["actual_departure"].apply(to_minutes)

delay = actual_dep_min - sched_dep_min
# handle overnight wrap-around (actual just after midnight vs scheduled just before)
delay = delay.apply(lambda d: d + 1440 if pd.notna(d) and d < -600 else d)
delay = delay.apply(lambda d: d - 1440 if pd.notna(d) and d > 600 else d)
df["delay_minutes"] = delay
df["is_delayed"] = df["delay_minutes"] > 5

df[["scheduled_departure", "actual_departure", "delay_minutes", "is_delayed"]].head()

,scheduled_departure,actual_departure,delay_minutes,is_delayed
0,20:25,20:25,0.0,False
1,14:50,15:12,22.0,True
2,08:15,08:15,0.0,False
3,17:45,17:45,0.0,False
4,13:00,13:40,40.0,True


In [18]:
# Net revenue derived column
df["net_revenue_inr"] = (df["fare_inr"] - df["discount_inr"]).clip(lower=0)

# Age group bins
df["age_group"] = pd.cut(
    df["age"],
    bins=[0, 25, 35, 45, 60, 100],
    labels=["18-25", "26-35", "36-45", "46-60", "60+"]
)

df[["net_revenue_inr", "age_group"]].head()

,net_revenue_inr,age_group
0,153.0,18-25
1,127.0,18-25
2,215.0,18-25
3,310.0,18-25
4,202.0,18-25


16d — ID / unique-key columns:

validate uniqueness and format consistency.

In [19]:
print("trip_id unique format check (all match TRP######):",
      df["trip_id"].str.match(r"^TRP\d{6}$").all())
print("customer_id unique format check (all match CUST#####):",
      df["customer_id"].str.match(r"^CUST\d{5}$").all())
print("route_id unique format check (all match RT..###):",
      df["route_id"].str.match(r"^RT[A-Z]{2}\d{3}$").all())

trip_id unique format check (all match TRP######): True
customer_id unique format check (all match CUST#####): True
route_id unique format check (all match RT..###): True


16e — Cross-column & general checks:

logical consistency (arrival not before departure), referential checks (cancellation_reason only present for Cancelled/No-show trips), remove the already-identified duplicates, and reset the index.

In [20]:
# Drop the duplicate rows identified in Step 5
before = len(df)
df = df.drop_duplicates(subset="trip_id", keep="first")
print(f"Dropped {before - len(df)} duplicate rows")

# Referential check: cancellation_reason should only exist for Cancelled / No-show trips
bad_reason = df[(df["cancellation_reason"].notna()) &
                 (~df["trip_status"].isin(["Cancelled", "No-show"]))]
print(f"Rows with cancellation_reason but status not Cancelled/No-show: {len(bad_reason)}")

# Logical check: delay_minutes should be NaN exactly when trip_status == Cancelled
mismatch = df[(df["trip_status"] == "Cancelled") & (df["delay_minutes"].notna())]
print(f"Cancelled trips with a non-null delay_minutes (should be 0): {len(mismatch)}")

df = df.reset_index(drop=True)
print(f"\nFinal cleaned shape: {df.shape}")

Dropped 58 duplicate rows
Rows with cancellation_reason but status not Cancelled/No-show: 0
Cancelled trips with a non-null delay_minutes (should be 0): 0

Final cleaned shape: (3200, 36)


Step 17 — Convert Into Final cleaned.csv File

Export the cleaned, prepared dataset as a single cleaned.csv to use as the base for all further analysis.

In [21]:
CLEANED_PATH = "cityflo_bus_service_metro_cities_cleaned.csv"
df.to_csv(CLEANED_PATH, index=False)
print(f"Saved cleaned dataset -> {CLEANED_PATH}  |  shape={df.shape}")

Saved cleaned dataset -> cityflo_bus_service_metro_cities_cleaned.csv  |  shape=(3200, 36)
